# India Electric Vehicle Market Analysis (2019–2025)
### An inclusive data analysis of India's EV passenger car market

**Datasets used:**
- `ev_sales.csv` — Annual model-level sales data (2019–2025)
- `ev_models.csv` — Technical specs & pricing per model
- `state_sales.csv` — State-wise EV registrations (2024)
- `policy.csv` — Government policy milestones & budgets

**Key questions answered:**
1. How has India's EV market grown year-over-year?
2. Which brands and models dominate the market?
3. Is there a price-range sweet spot driving adoption?
4. Which states are leading EV adoption?
5. How have government policies correlated with sales growth?

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor': '#FFFFFF',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
})

PALETTE = ['#185FA5', '#1D9E75', '#D85A30', '#BA7517', '#7F77DD', '#C44569', '#2C3E50']
sns.set_palette(PALETTE)

print("Libraries loaded ")

In [ ]:
# Load datasets
sales = pd.read_csv('data/ev_sales.csv')
models = pd.read_csv('data/ev_models.csv')
states = pd.read_csv('data/state_sales.csv')
policy = pd.read_csv('data/policy.csv')

# --- Clean sales ---
sales.columns = sales.columns.str.strip()
sales['Car_Brand'] = sales['Car_Brand'].str.strip()
sales['Model'] = sales['Model'].str.strip()
sales['Segment'] = sales['Segment'].str.strip().str.title()
sales['Segment'] = sales['Segment'].replace('-', 'Other')

# --- Clean models ---
models.columns = models.columns.str.strip()
models['Car_Brand'] = models['Car_Brand'].str.strip()
models['Model'] = models['Model'].str.strip()
# Fix casing mismatch
models['Model'] = models['Model'].replace('ClavisEV', 'ClavisEv')
models.rename(columns={
    'price (Lakhs) (on road)': 'Price_Lakh',
    'battery (kWh)': 'Battery_kWh',
    'launch_year': 'Launch_Year'
}, inplace=True)

# --- Clean states ---
states['Units_Sold'] = states['Units_Sold'].astype(str).str.replace(',', '').astype(int)

# --- Clean policy ---
policy.columns = policy.columns.str.strip()
policy.rename(columns={'Budget (in crore)': 'Budget_Crore'}, inplace=True)

print("Sales shape:", sales.shape)
print("Models shape:", models.shape)
print("States shape:", states.shape)
print("Policy shape:", policy.shape)
print("\nData loaded & cleaned ")

In [ ]:
# Quick peek at each dataset
print("=== EV SALES (first 5 rows) ===")
display(sales.head())
print("\n=== EV MODELS SPECS ===")
display(models.head())
print("\n=== STATE SALES ===")
display(states)
print("\n=== POLICY TIMELINE ===")
display(policy)

## 2. Exploratory Data Analysis

In [ ]:
# Basic stats
print("SUMMARY STATISTICS\n")
print(f"Years covered: {sales['Year'].min()} – {sales['Year'].max()}")
print(f"Total brands: {sales['Car_Brand'].nunique()}")
print(f"Total models: {sales['Model'].nunique()}")
print(f"Total units sold (all years): {sales['Units_Sold'].sum():,}")
print(f"\nTop brand by total sales: {sales.groupby('Car_Brand')['Units_Sold'].sum().idxmax()}")
print(f"Top model by total sales: {sales.groupby('Model')['Units_Sold'].sum().idxmax()}")
print(f"\nSegment breakdown:")
display(sales.groupby('Segment')['Units_Sold'].sum().sort_values(ascending=False))

In [ ]:
# Missing values check
print("Missing values per dataset:")
for name, df in [('Sales', sales), ('Models', models), ('States', states), ('Policy', policy)]:
    print(f"\n{name}:")
    print(df.isnull().sum())

## 3. Market Growth Analysis

In [ ]:
# Total EV sales by year
yearly = sales.groupby('Year')['Units_Sold'].sum().reset_index()
yearly['YoY_Growth_%'] = yearly['Units_Sold'].pct_change() * 100
yearly['YoY_Growth_%'] = yearly['YoY_Growth_%'].round(1)
print("Year-over-year EV sales:")
display(yearly)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('India EV Market Growth (2019–2025)', fontsize=16, fontweight='bold', y=1.02)

# Chart 1: Total sales bar
ax1 = axes[0]
bars = ax1.bar(yearly['Year'], yearly['Units_Sold'], color=PALETTE[0], alpha=0.85, width=0.6)
for bar, val in zip(bars, yearly['Units_Sold']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
             f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_title('Total EV Units Sold Per Year')
ax1.set_xlabel('Year')
ax1.set_ylabel('Units Sold')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Chart 2: YoY Growth
ax2 = axes[1]
yoy = yearly.dropna(subset=['YoY_Growth_%'])
colors = [PALETTE[1] if v > 0 else '#E74C3C' for v in yoy['YoY_Growth_%']]
bars2 = ax2.bar(yoy['Year'], yoy['YoY_Growth_%'], color=colors, alpha=0.85, width=0.6)
for bar, val in zip(bars2, yoy['YoY_Growth_%']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{val:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_title('Year-over-Year Growth Rate')
ax2.set_xlabel('Year')
ax2.set_ylabel('Growth (%)')
ax2.axhline(0, color='gray', linewidth=0.8)

plt.tight_layout()
plt.savefig('data/fig1_market_growth.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved ")

**Insight:** The Indian EV passenger car market exploded after 2021, growing ~400% between 2021 and 2022. Growth moderated in 2023-24 as the market matured, but 2025 shows a fresh surge driven by Mahindra's new BE6/XEV9E lineup and MG Windsor's strong sales.

## 4. Brand Market Share Analysis

In [ ]:
# Brand share by year
brand_year = sales.groupby(['Year', 'Car_Brand'])['Units_Sold'].sum().reset_index()
brand_year_pct = brand_year.copy()
brand_year_pct['Share_%'] = brand_year_pct.groupby('Year')['Units_Sold'].transform(lambda x: x / x.sum() * 100)

# Overall brand share (all years)
brand_total = sales.groupby('Car_Brand')['Units_Sold'].sum().sort_values(ascending=False).reset_index()
brand_total['Share_%'] = (brand_total['Units_Sold'] / brand_total['Units_Sold'].sum() * 100).round(1)
print("Overall brand share (2019–2025):")
display(brand_total)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Brand Market Share Analysis', fontsize=16, fontweight='bold')

# Chart 1: Stacked area - brand share over years
ax1 = axes[0]
pivot = brand_year_pct.pivot_table(index='Year', columns='Car_Brand', values='Share_%', fill_value=0)
pivot.plot(kind='area', stacked=True, ax=ax1, alpha=0.8,
           color=PALETTE[:len(pivot.columns)])
ax1.set_title('Brand Share Over Time (%)')
ax1.set_xlabel('Year')
ax1.set_ylabel('Market Share (%)')
ax1.legend(loc='upper left', fontsize=8, framealpha=0.7)
ax1.set_ylim(0, 100)

# Chart 2: Donut chart - overall share
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    brand_total['Units_Sold'],
    labels=brand_total['Car_Brand'],
    autopct='%1.1f%%',
    startangle=140,
    colors=PALETTE[:len(brand_total)],
    pctdistance=0.82,
    wedgeprops=dict(width=0.55)
)
for at in autotexts:
    at.set_fontsize(8)
ax2.set_title('Overall Brand Share (2019–2025)')

plt.tight_layout()
plt.savefig('data/fig2_brand_share.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Brand sales trend lines
fig, ax = plt.subplots(figsize=(12, 6))

top_brands = brand_total['Car_Brand'].head(6).tolist()
for i, brand in enumerate(top_brands):
    data = brand_year[brand_year['Car_Brand'] == brand]
    ax.plot(data['Year'], data['Units_Sold'], marker='o', linewidth=2.5,
            markersize=6, label=brand, color=PALETTE[i])
    # Label last point
    last = data.iloc[-1]
    ax.annotate(f"{int(last['Units_Sold']):,}",
                xy=(last['Year'], last['Units_Sold']),
                xytext=(5, 0), textcoords='offset points',
                fontsize=8, color=PALETTE[i])

ax.set_title('Brand-wise Sales Trend (2019–2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Units Sold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('data/fig3_brand_trends.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** Tata Motors has dominated India's EV market since 2020 with affordable options like the Nexon EV, Tiago EV, and Punch EV. However, 2025 marks a notable shift — Mahindra's BE6 launch rapidly closed the gap, signaling the start of genuine competition in the premium segment.

## 5. Model-level Analysis

In [ ]:
# Top 10 models by total sales
model_total = sales.groupby(['Car_Brand', 'Model'])['Units_Sold'].sum().reset_index()
model_total = model_total.sort_values('Units_Sold', ascending=False).head(10)
model_total['Model_Label'] = model_total['Car_Brand'] + ' ' + model_total['Model']

fig, ax = plt.subplots(figsize=(11, 6))
colors = [PALETTE[['Tata','MG','Mahindra','Hyundai','BYD','Citroen','Kia','VinFast','Others'].index(b) % len(PALETTE)]
          for b in model_total['Car_Brand']]
bars = ax.barh(model_total['Model_Label'][::-1], model_total['Units_Sold'][::-1],
               color=colors[::-1], alpha=0.87)
for bar, val in zip(bars, model_total['Units_Sold'][::-1]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9, fontweight='bold')
ax.set_title('Top 10 EV Models by Total Sales (2019–2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Units Sold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('data/fig4_top_models.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Segment analysis
seg_year = sales[sales['Segment'] != 'Other'].groupby(['Year', 'Segment'])['Units_Sold'].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Segment Analysis', fontsize=14, fontweight='bold')

# Segment totals
seg_total = sales[sales['Segment'] != 'Other'].groupby('Segment')['Units_Sold'].sum().sort_values(ascending=True)
axes[0].barh(seg_total.index, seg_total.values, color=PALETTE[:len(seg_total)], alpha=0.85)
for i, (seg, val) in enumerate(zip(seg_total.index, seg_total.values)):
    axes[0].text(val + 200, i, f'{val:,}', va='center', fontsize=9)
axes[0].set_title('Total Sales by Segment (All Years)')
axes[0].set_xlabel('Units Sold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Segment over time
seg_pivot = seg_year.pivot_table(index='Year', columns='Segment', values='Units_Sold', fill_value=0)
seg_pivot.plot(kind='bar', ax=axes[1], alpha=0.85, color=PALETTE[:len(seg_pivot.columns)], width=0.7)
axes[1].set_title('Segment Sales by Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Units Sold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[1].legend(fontsize=8)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('data/fig5_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Price vs Range Analysis (EV Value Positioning)

In [ ]:
# Merge sales with model specs
merged = pd.merge(sales, models[['Car_Brand', 'Model', 'Price_Lakh', 'Battery_kWh', 'Segment']],
                  on=['Car_Brand', 'Model'], how='left')

# Filter to models with price data, exclude Others
specs_clean = models[models['Car_Brand'] != 'Others'].copy()
print(f"Models with spec data: {len(specs_clean)}")
display(specs_clean[['Car_Brand', 'Model', 'Price_Lakh', 'Battery_kWh', 'Segment']].sort_values('Price_Lakh'))

In [ ]:
# Price vs Battery scatter (bubble sized by total sales)
model_sales = sales.groupby(['Car_Brand', 'Model'])['Units_Sold'].sum().reset_index()
spec_sales = pd.merge(specs_clean, model_sales, on=['Car_Brand', 'Model'], how='left')
spec_sales['Units_Sold'] = spec_sales['Units_Sold'].fillna(100)

brand_colors = {
    'Tata': PALETTE[0], 'MG': PALETTE[1], 'Mahindra': PALETTE[2],
    'BYD': PALETTE[3], 'Hyundai': PALETTE[4], 'Citroen': PALETTE[5],
    'Kia': '#2C3E50', 'VinFast': '#8E44AD'
}

fig, ax = plt.subplots(figsize=(13, 7))

for brand, group in spec_sales.groupby('Car_Brand'):
    color = brand_colors.get(brand, '#999')
    scatter = ax.scatter(
        group['Price_Lakh'], group['Battery_kWh'],
        s=group['Units_Sold'] / 15,
        color=color, alpha=0.75, label=brand, edgecolors='white', linewidths=0.8
    )
    for _, row in group.iterrows():
        ax.annotate(row['Model'], (row['Price_Lakh'], row['Battery_kWh']),
                    textcoords='offset points', xytext=(5, 4),
                    fontsize=7.5, color='#333')

# Affordable zone shading
ax.axvspan(0, 20, alpha=0.04, color='green', label='_Affordable zone')
ax.axvline(20, color='green', linestyle='--', alpha=0.3, linewidth=1)
ax.text(10, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 75, '← Affordable (<₹20L)',
        fontsize=8, color='green', alpha=0.6)

ax.set_title('Price vs Battery Capacity\n(bubble size = total units sold)', fontsize=13, fontweight='bold')
ax.set_xlabel('Price (₹ Lakh, on-road)')
ax.set_ylabel('Battery Capacity (kWh)')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('data/fig6_price_vs_battery.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Price segment bucketing
specs_clean2 = spec_sales.copy()
bins = [0, 12, 20, 35, 200]
labels = ['Budget (<₹12L)', 'Mid (₹12–20L)', 'Premium (₹20–35L)', 'Luxury (>₹35L)']
specs_clean2['Price_Segment'] = pd.cut(specs_clean2['Price_Lakh'], bins=bins, labels=labels)

price_seg = specs_clean2.groupby('Price_Segment', observed=True)['Units_Sold'].sum().reset_index()
price_seg['Share_%'] = (price_seg['Units_Sold'] / price_seg['Units_Sold'].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(price_seg['Price_Segment'], price_seg['Units_Sold'],
              color=[PALETTE[1], PALETTE[0], PALETTE[2], PALETTE[3]], alpha=0.85)
for bar, row in zip(bars, price_seg.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'{row.Units_Sold:,}\n({row._3}%)', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Sales Volume by Price Segment', fontsize=13, fontweight='bold')
ax.set_xlabel('Price Segment')
ax.set_ylabel('Total Units Sold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('data/fig7_price_segments.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey finding: Mid-range EVs (₹12–20L) dominate volume — confirming affordability as the #1 adoption driver")

## 7. State-wise Adoption Analysis

In [ ]:
states_sorted = states.sort_values('Units_Sold', ascending=True)
states_sorted['Share_%'] = (states_sorted['Units_Sold'] / states_sorted['Units_Sold'].sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('State-wise EV Adoption (2024)', fontsize=14, fontweight='bold')

# Horizontal bar
color_gradient = plt.cm.Blues(np.linspace(0.35, 0.85, len(states_sorted)))
bars = axes[0].barh(states_sorted['State'], states_sorted['Units_Sold'],
                    color=color_gradient, alpha=0.9)
for bar, row in zip(bars, states_sorted.itertuples()):
    axes[0].text(bar.get_width() + 400, bar.get_y() + bar.get_height()/2,
                 f'{row.Units_Sold:,}', va='center', fontsize=9)
axes[0].set_title('EV Registrations by State')
axes[0].set_xlabel('Units Sold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Pie
states_pie = states.sort_values('Units_Sold', ascending=False)
top5 = states_pie.head(5)
others_val = states_pie.iloc[5:]['Units_Sold'].sum()
pie_data = list(top5['Units_Sold']) + [others_val]
pie_labels = list(top5['State']) + ['Others']
axes[1].pie(pie_data, labels=pie_labels, autopct='%1.1f%%', startangle=140,
            colors=plt.cm.Blues(np.linspace(0.4, 0.85, 6)),
            wedgeprops=dict(width=0.55), pctdistance=0.8)
axes[1].set_title('Top 5 States Share')

plt.tight_layout()
plt.savefig('data/fig8_state_adoption.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Policy Impact on EV Adoption

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))

# Sales bars
ax1.bar(yearly['Year'], yearly['Units_Sold'], color=PALETTE[0], alpha=0.6, label='EV Units Sold', width=0.5)
ax1.set_xlabel('Year')
ax1.set_ylabel('Units Sold', color=PALETTE[0])
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.tick_params(axis='y', labelcolor=PALETTE[0])

# Policy budget on secondary axis
ax2 = ax1.twinx()
ax2.plot(policy['Year'], policy['Budget_Crore'], 'o--', color=PALETTE[2],
         linewidth=2, markersize=8, label='Policy Budget (₹ Cr)')
ax2.set_ylabel('Policy Budget (₹ Crore)', color=PALETTE[2])
ax2.tick_params(axis='y', labelcolor=PALETTE[2])
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Policy annotations
for _, row in policy.iterrows():
    ax1.axvline(x=row['Year'], color='gray', linestyle=':', alpha=0.4, linewidth=1)
    y_pos = yearly[yearly['Year'] == row['Year']]['Units_Sold'].values
    y_val = y_pos[0] if len(y_pos) > 0 else 5000
    ax1.text(row['Year'], y_val + 3000,
             row['Policy Name'].replace(' ', '\n'), ha='center', fontsize=6.5,
             color='#555', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_title('Government Policy Spend vs EV Sales Growth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig9_policy_impact.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** The FAME India II scheme (₹11,500 Cr, 2019) was the single biggest policy catalyst. Sales didn't respond immediately — there's a 2 year lag but the 2022 explosion in EV sales directly followed FAME II subsidies reaching consumers. The PLI schemes for batteries and manufacturing are expected to drive a structural cost reduction through 2025–27.

## 9. Competitive Landscape: 2025 DeepDive

In [ ]:
# 2025 brand breakdown
sales_2025 = sales[sales['Year'] == 2025].copy()
brand_2025 = sales_2025.groupby('Car_Brand')['Units_Sold'].sum().sort_values(ascending=False).reset_index()
brand_2025['Share_%'] = (brand_2025['Units_Sold'] / brand_2025['Units_Sold'].sum() * 100).round(1)

print("2025 Brand Standings:")
display(brand_2025)

total_2025 = sales_2025['Units_Sold'].sum()
total_2024 = sales[sales['Year'] == 2024]['Units_Sold'].sum()
print(f"\nTotal 2024 sales: {total_2024:,}")
print(f"Total 2025 sales: {total_2025:,}")
print(f"YoY growth: {((total_2025/total_2024)-1)*100:.1f}%")

In [ ]:
# 2024 vs 2025 brand comparison
brand_24 = sales[sales['Year'] == 2024].groupby('Car_Brand')['Units_Sold'].sum().reset_index()
brand_24.columns = ['Car_Brand', '2024']
brand_25 = sales[sales['Year'] == 2025].groupby('Car_Brand')['Units_Sold'].sum().reset_index()
brand_25.columns = ['Car_Brand', '2025']
comp = pd.merge(brand_24, brand_25, on='Car_Brand', how='outer').fillna(0)
comp['Change'] = comp['2025'] - comp['2024']
comp['Change_%'] = ((comp['2025'] / comp['2024'].replace(0, np.nan)) - 1) * 100
comp = comp.sort_values('2025', ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(comp))
w = 0.35
ax.bar(x - w/2, comp['2024'], w, label='2024', color=PALETTE[0], alpha=0.7)
ax.bar(x + w/2, comp['2025'], w, label='2025', color=PALETTE[1], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(comp['Car_Brand'], rotation=15)
ax.set_title('Brand Sales: 2024 vs 2025 Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('Units Sold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
plt.tight_layout()
plt.savefig('data/fig10_2024_vs_2025.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Key Findings & Conclusions

In [ ]:
print("=" * 60)
print("  Indian Market — Key Findings in (2019–2025)")
print("=" * 60)

total_all = sales['Units_Sold'].sum()
tata_total = sales[sales['Car_Brand'] == 'Tata']['Units_Sold'].sum()
best_model = sales.groupby('Model')['Units_Sold'].sum().idxmax()
best_model_sales = sales.groupby('Model')['Units_Sold'].sum().max()
peak_growth = yearly['YoY_Growth_%'].max()
peak_year = yearly.loc[yearly['YoY_Growth_%'].idxmax(), 'Year']

print(f"""
1. Market Size 
   Total EV passenger cars sold (2019–2025): {total_all:,}
   2025 alone: {total_2025:,} units — highest single year

2. Market Leader 
   Tata Motors: {tata_total:,} units = {tata_total/total_all*100:.1f}% of all EVs sold
   Dominant since 2020 due to early-mover advantage & affordable pricing

3. Best Selling Model 
   {best_model}: {best_model_sales:,} total units

4. Growth Boom 
   Peak YoY growth: {peak_growth:.0f}% in {peak_year}
   Driven by FAME II subsidies reaching consumers + Tata's Nexon EV

5. Affordability 
   ~60%+ of sales come from models priced under ₹20L
   Budget segment (<₹12L) opened by Tiago EV & MG Comet

6. The 2025 Shift
   Mahindra BE6 launched → fastest-selling EV ever in India
   MG Windsor EV surpassed ZS EV within 1 year
   Market moving from single-brand dominance to multi-brand competition

7. Geographic Concentration 
   Top 3 states (Maharashtra, Karnataka, Kerala) = ~48% of all EV sales
   Tier-2 city adoption still nascent — major growth runway ahead

8. Policy Correlation
   Every major sales inflection point followed a government policy event
   PM E-Drive (2024) + PLI for batteries expected to drive 2026–27 growth
""")
print("=" * 60)

---
## Data Sources
- Sales data compiled from VAHAN Dashboard (vahan.parivahan.gov.in), SIAM Annual Reports, and industry publications (ET Auto, CarDekho)
- Model specs from brand official websites, CarWale, EvoIndia and TeamBHP 
- State-wise data from VAHAN (2024 snapshot)
- Policy data from Ministry of Heavy Industries (PIB releases)

---
*Analysis by: Gopal Vashistha | Tools: Python, pandas, matplotlib, seaborn*